### installing all the packages.

In [1]:
!pip install -q torch datasets transformers peft trl bitsandbytes accelerate

#installing all the required libraries for fine-tuning the model
#torch: for model training and inference
#datasets: for loading and processing datasets
#transformers: for using pre-trained transformer models
#peft: for parameter-efficient fine-tuning of transformer models
#trl: for training and evaluating transformer models
#bitsandbytes: for efficient training with 8-bit optimizers
#accelerate: to place the model on the GPU for faster training and inference

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 22.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.3 MB/s eta 0:00:00:00:0100:01


### Importing all library 

In [2]:
import os
import torch
#hugging face datasets library to load the dataset for training pourpose can laod the dataset from local files or from hugging face.
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, #load llm model from (llaMA, mistral etc) from hugging face model hub
    AutoTokenizer,        #convert the text into tokens so that the model can understand and process it.
    BitsAndBytesConfig,   #configure the model to use 4-bits/8-bit quantization to reduce memory usage and speed up training.
    HfArgumentParser,      
    TrainingArguments, 
    pipeline,             # pipeline is a high-level API provided by the transformers library that allows you to easily perform various NLP tasks such as text generation, sentiment analysis, and question answering using pre-trained models.
    logging,              # logging is a module in the transformers library that provides a way to log messages and information during the training and evaluation of models. It allows you to track the progress of your training, monitor metrics, and debug issues.

)
from peft import LoraConfig, get_peft_model 
from trl import SFTTrainer

### Setting up model parameters

In [ ]:
#seleting model from huggineface model hub for fine_tuning.
model_name = "NousResearch/Llama-2-7b-chat-hf"

# selecting the dataset from the hugging face model hub for fine_tuning the model.
dataset_name = "mlabonne/guanaco-llama2-1k"

# Fine-tuned model name
new_model = "Llama-2-7b-chat-finetune"

In [4]:
dataset = load_dataset(dataset_name, split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

### LoRA practice to fine-tune LLM.
LoRA (Low-Rank Adaptation) is a procedure for fine-tuning a neural network 
in which we freeze the weight matrix of the LLM so that it remains 
untouched, and a new small matrix is introduced in the fine-tuning process.

The weight matrix consists of the weights of the neural network, and it 
indicates how one neuron affects another in the neural network.

The mental model behind introducing new, smaller matrices and freezing 
the older ones is to make sure we only train the newer ones and preserve 
the older ones — so that all the training happens on the newly introduced 
weight matrices. This is done to preserve general-purpose intelligence, 
while gaining domain-specific intelligence in the new weight matrices.

To completely understand LoRA we need to understand to concept of adapters.
Adapters are the small, aditional trainble components inserted into the pertrined
neural network designed to let model learn a new task without affecting the pretrained
weights, A and B (the two new matrices) are exactly what's called an "adapter." 


In [5]:
# setting quantization configuration for the model to use 4-bits/8-bit quantization
# to reduce memory usage and speed up training.
# lora dimension attention
#R is the rank of the low-rank matrices used in the LoRA method. 
#It controls the capacity of the adaption. 
#A higher R allows for more expressive adaptations but increases the computational cost.
#different values of R can be exerimented with to find the optimal balance between model
#performance and traning efficiency.
#example: R=4, 8, 16, 32, 64, 128"""
# inceasign the R, increaes the model's capacity to learn complex patterns from the data,
# but it also increases the number of parameters.
# in simple terms, a higher R means larger matrices with parameters.
lora_r = 64


# introducing a scaling factor to control the impact of the LoRA layers on the model's output.
# it indicates how much the loRA layers should influence the model's predictions.
# higher alpha values give more weight to the LoRA layers, while lower values reduce their influence.
Lora_alpha = 16


# setting a dropout rate for the LoRA layers to prevent overfitting during training. 
# it is regularization technique that randomly sets a fraction of the input units to zero during training
# During each forward pass, a certain percentage of the neurons in the loRA layers are "dropped out" hence 
# generalizes better.
lora_dropout = 0.1


# bits and bytes congfiguration for the model to use 4-bits/8-bit quantization to reduce memory usage and 
# speed up training.
# Activate 4-bit precision base model loading.
use_4bit = True


# configuration setting for the model to use 4-bits/8bit quantization to reduce memory usage and speed up training.
# BitsAndBytesConfig that controls the precision used during the actual math operations.
# When you load a model in 4-bit (via load_in_4bit=True), the weights sit in GPU memory compressed to 4 bits each. 
# That's great for memory savings — a 7B model that would take ~14GB in fp16 now takes ~3.5-4GB.
bnb_4bit_compute_dtype = torch.float16


#nf4 is a quantization type that uses 4 bits to represent each weight in the model.
bnb_4bit_quant_type = "nf4" 

# Activte nested quantization for the model to use 4-bits/8-bit quantization to reduce memory usage and 
# speed up training.
use_nested_quant = False




In [6]:
compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

: 

In [9]:
peft_params = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [11]:
training_params = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="tensorboard"
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'group_by_length'

: 